# Timing: (k tokens, last-layer MLP output) @ cached unembedding matrix

We have a cached unembedding matrix $W_U \in \mathbb{R}^{V \times d}$ for Llama-3.1-8B
($V$ = vocab size, $d$ = model dim). The last-layer MLP output for a token is a
vector in $\mathbb{R}^{d}$. Here we just use **random** vectors as stand-ins.

For $k$ tokens we form $X \in \mathbb{R}^{d \times k}$ and time the logit projection
$$ L = W_U \, X \in \mathbb{R}^{V \times k}. $$
We do this for $k = 1, 5, 30$.

In [1]:
import time
from pathlib import Path
import torch

# Cached unembedding matrix produced by APICache.get_unembedding_matrix (float32, contiguous).
CACHE = Path("../real/backend/results_cache/unembedding_matracies/meta-llama%2FLlama-3.1-8B")
assert CACHE.exists(), CACHE

W_U = torch.load(CACHE)  # (V, d)
W_U = W_U.float().contiguous()
V, d = W_U.shape
device = "cuda" if torch.cuda.is_available() else "cpu"
W_U = W_U.to(device)
print(f"W_U shape = {tuple(W_U.shape)}  dtype={W_U.dtype}  device={device}")

W_U shape = (128256, 4096)  dtype=torch.float32  device=cpu


In [2]:
def time_matmul(k: int, n_repeats: int = 20, n_warmup: int = 3) -> dict:
    """Time W_U @ X for X = random (d, k). Returns timing stats in milliseconds.

    Uses CUDA events when on GPU (accounts for async), perf_counter on CPU.
    CLAUDE_WRITTEN
    """
    X = torch.randn(d, k, device=device, dtype=W_U.dtype)
    use_cuda = device == "cuda"

    # Warmup (kernel autotuning / allocation).
    for _ in range(n_warmup):
        _ = W_U @ X
    if use_cuda:
        torch.cuda.synchronize()

    times_ms = []
    for _ in range(n_repeats):
        if use_cuda:
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            L = W_U @ X
            end.record()
            torch.cuda.synchronize()
            times_ms.append(start.elapsed_time(end))
        else:
            t0 = time.perf_counter()
            L = W_U @ X
            times_ms.append((time.perf_counter() - t0) * 1e3)
    assert L.shape == (V, k), L.shape
    t = torch.tensor(times_ms)
    return {"k": k, "mean_ms": t.mean().item(), "std_ms": t.std().item(), "min_ms": t.min().item()}

In [ ]:
results = [time_matmul(k,1) for k in (1, 5, 30)]
print(f"{'k':>4} {'mean (ms)':>12} {'std (ms)':>12} {'min (ms)':>12}")
for r in results:
    print(f"{r['k']:>4} {r['mean_ms']:>12.4f} {r['std_ms']:>12.4f} {r['min_ms']:>12.4f}")

   k    mean (ms)     std (ms)     min (ms)
   1     112.8201          nan     112.8201


C:\Users\wildn\AppData\Local\Temp\ipykernel_8732\3905820589.py:32: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1879.)
  return {"k": k, "mean_ms": t.mean().item(), "std_ms": t.std().item(), "min_ms": t.min().item()}


In [8]:
results = [time_matmul(k,1) for k in (30,)]
print(f"{'k':>4} {'mean (ms)':>12} {'std (ms)':>12} {'min (ms)':>12}")
for r in results:
    print(f"{r['k']:>4} {r['mean_ms']:>12.4f} {r['std_ms']:>12.4f} {r['min_ms']:>12.4f}")

   k    mean (ms)     std (ms)     min (ms)
  30     244.2009          nan     244.2009


C:\Users\wildn\AppData\Local\Temp\ipykernel_8732\3905820589.py:32: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1879.)
  return {"k": k, "mean_ms": t.mean().item(), "std_ms": t.std().item(), "min_ms": t.min().item()}
